In [1]:
from system_check import *
CUDA_state_print_limited()

CUDA доступна: True
Количество GPU: 4


In [2]:
import torch

def get_device() -> str:
    """
    Возвращает 'cuda' если GPU доступен, иначе 'cpu'.
    """
    return "cuda:1" if torch.cuda.is_available() else "cpu"

In [3]:
device = get_device()
print(f"Используемое устройство: {device}")

Используемое устройство: cuda:1


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, gc

# Очистка памяти от предыдущих моделей
gc.collect()
torch.cuda.empty_cache()

# Загрузка ТОЛЬКО на CPU (без accelerate/device_map)
model_id = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token  # критически важно для Qwen

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,      # экономия памяти
    low_cpu_mem_usage=True,         # загрузка без переполнения ОЗУ
    trust_remote_code=False         # не требуется для Qwen2.5
).to(device)  # ← ЯВНОЕ перемещение на CPU

print("✅ Модель загружена. Память:", sum(p.numel() for p in model.parameters()) / 1e9, "B параметров")

/home/pmartynyuk/UnTIE project/untie_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.55s/it]


✅ Модель загружена. Память: 3.085938688 B параметров


In [5]:
def answer_question_qwen(
    context: str, 
    question: str, 
    max_new_tokens: int = 150,
    max_context_tokens: int = 16384  # ← НАСТРОЙКА: лимит контекста
) -> str:
    """
    Получает ответ от Qwen2.5-3B с поддержкой длинного контекста.
    
    Args:
        context: Текст для анализа (до 128K токенов)
        question: Вопрос к тексту
        max_new_tokens: Макс. длина генерируемого ответа
        max_context_tokens: Лимит токенов контекста (рекомендуется 8192–16384 для ваших данных)
    """
    # === 1. Обрезка контекста ДО формирования промпта ===
    tokens = tokenizer.encode(context, add_special_tokens=False)
    if len(tokens) > max_context_tokens:
        # Сохраняем КОНЕЦ текста (часто содержит выводы/ответы)
        tokens = tokens[-max_context_tokens:]
        context = tokenizer.decode(tokens, skip_special_tokens=True)
    
    # === 2. Формирование промпта ===
    messages = [{
        "role": "user",
        "content": (
            "Ответь строго по тексту ниже. Если информации нет — напиши 'Не указано в тексте'.\n\n"
            f"Текст:\n{context}\n\n"
            f"Вопрос: {question}"
        )
    }]
    
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # === 3. Токенизация с расширенным лимитом ===
    # Важно: max_length должен быть >= max_context_tokens + длина вопроса + системные токены
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_context_tokens + 512  # +512 на вопрос и системные токены
    ).to(device)
    
    # === 4. Генерация ===
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False
        )
    
    # === 5. Извлечение ответа ===
    response_tokens = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(response_tokens, skip_special_tokens=True)
    
    return response.strip()

In [7]:
import pandas as pd
import time
from tqdm import tqdm

In [9]:
df_res = pd.read_csv('/home/pmartynyuk/UnTIE project/datasets/ruserrc_task.csv', 
                    encoding='utf-8', 
                    sep=';')
df_res

,text,Task_aspects,Task_count,text_clean
0,В рамках проекта по <Task> автоматизации раб...,['автоматизации работы с поэтическими текстами...,2,В рамках проекта по автоматизации работы с ...
1,Проблема <Task> своевременного предупреждения...,['своевременного предупреждения населения об о...,2,Проблема своевременного предупреждения насел...
2,Работа посвящена <Contrib> описанию структур...,['генотипирования вируса клещевого энцефалита ...,2,Работа посвящена описанию структуры информа...
3,Для эффективного <Task> освоения залежей угл...,['освоения залежей углеводородов'],1,Для эффективного освоения залежей углеводор...
4,При <Task> формировании единого информационно...,['формировании единого информационного простра...,1,При формировании единого информационного про...
...,...,...,...,...
163,Рассматриваются проблемы <Task> проектирован...,['проектирования интеллектуальных систем управ...,1,Рассматриваются проблемы проектирования инт...
164,"Планирование землепользования , охрана окружаю...",['поддержки принятия решений'],1,"Планирование землепользования , охрана окружаю..."
165,Во многих <Task> задачах текстовой классифик...,['задачах текстовой классификации'],1,Во многих задачах текстовой классификации ...
166,<Task> Выбор оптимального расположения логис...,['Выбор оптимального расположения логистически...,2,Выбор оптимального расположения логистическ...


In [19]:
df_res = df_res.drop(columns=["qwen2.5-3B_answer", "time"])

In [20]:
# Создаём колонку для ответов, если её нет
if 'qwen2.5-3B_answer' not in df_res.columns:
    df_res['qwen2.5-3B_answer'] = pd.NA
    df_res['time'] = pd.NA

# Вопрос для всех текстов
question = "Какая задача была решена? Дай краткий ответ без объяснения"

# Прогресс-бар
pbar = tqdm(
    total=len(df_res),
    desc="Обработка текстов Qwen2.5-3B",
    unit="текст",
    ncols=80
)

# Счётчики
success_count = 0
error_count = 0

try:
    for idx in range(len(df_res)):
        # Пропускаем уже обработанные строки
        if pd.notna(df_res.loc[idx, 'qwen2.5-3B_answer']) and df_res.loc[idx, 'qwen2.5-3B_answer'] != '':
            success_count += 1
            pbar.update(1)
            continue
        
        context = df_res.loc[idx, 'text_clean']
        
        try:
            # Получаем ответ через вашу функцию
            start_time = time.time()
            answer = answer_question_qwen(
                context=context,
                question=question,
                max_new_tokens=100,        # короткий ответ как в примере
                max_context_tokens=16384   # полный контекст для ваших длинных текстов
            )
            elapsed = time.time() - start_time
            
            # Сохраняем ответ
            df_res.loc[idx, 'qwen2.5-3B_answer'] = answer
            df_res.loc[idx, 'time'] = elapsed
            success_count += 1
            
            # Обновляем прогресс-бар
            pbar.set_postfix({
                'успех': success_count,
                'ошибок': error_count,
                'время': f"{elapsed:.1f}с"
            })
            
        except Exception as e:
            error_count += 1
            error_msg = f"[ERROR] {str(e)[:60]}"
            df_res.loc[idx, 'qwen2.5-3B_answer'] = error_msg
            df_res.loc[idx, 'time'] = 0
            pbar.set_postfix({'ошибок': error_count})
            print(f"\n⚠️ Строка {idx}: {error_msg}")
        
        pbar.update(1)
        
        # Пауза для стабильности на CPU
        time.sleep(0.3)
        
        # Промежуточное сохранение каждые 10 строк
        if (idx + 1) % 10 == 0:
            df_res.to_csv('rus_df_res_with_qwen_answers.csv', index=False)
            pbar.write(f" 💾 Сохранено {idx + 1}/{len(df_res)} строк")

except KeyboardInterrupt:
    pbar.close()
    print("\n\n🛑 Обработка прервана. Сохраняем результаты...")
    df_res.to_csv('rus_df_res_with_qwen_answers.csv', index=False)
    print("✅ Промежуточные результаты сохранены в 'df_res_with_qwen_answers.csv'")
    raise

finally:
    pbar.close()

# Финальное сохранение
df_res.to_csv('rus_df_res_with_qwen_answers.csv', index=False)

# Итоговая статистика
print("\n" + "="*70)
print("✅ ОБРАБОТКА ЗАВЕРШЕНА")
print(f"   Всего строк:    {len(df_res)}")
print(f"   Успешно:        {success_count} ({success_count/len(df_res)*100:.1f}%)")
print(f"   Ошибок:         {error_count} ({error_count/len(df_res)*100:.1f}%)")
print(f"   Результаты:     df_res_with_qwen_answers.csv")
print("="*70)

# Показать примеры ответов
print("\n🔍 Примеры ответов:")
for i in range(min(5, len(df_res))):
    print(f"\nСтрока {i}:")
    print(f"   Ответ: {df_res.loc[i, 'qwen2.5-3B_answer'][:100]}...")

Обработка текстов Qwen2.5-3B:   6%| | 10/168 [00:24<07:30,  2.85s/текст, успех=1The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


 💾 Сохранено 10/168 строк


Обработка текстов Qwen2.5-3B:  12%| | 20/168 [00:45<05:04,  2.06s/текст, успех=2The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


 💾 Сохранено 20/168 строк


Обработка текстов Qwen2.5-3B:  18%|▏| 30/168 [01:11<05:09,  2.25s/текст, успех=3The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...

 💾 Сохранено 150/168 строк


Обработка текстов Qwen2.5-3B:  95%|▉| 160/168 [06:30<00:34,  4.29s/текст, успех=The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


 💾 Сохранено 160/168 строк


Обработка текстов Qwen2.5-3B:  99%|▉| 167/168 [06:55<00:02,  2.89s/текст, успех=The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Обработка текстов Qwen2.5-3B: 100%|█| 168/168 [06:57<00:00,  2.48s/текст, успех=


✅ ОБРАБОТКА ЗАВЕРШЕНА
   Всего строк:    168
   Успешно:        168 (100.0%)
   Ошибок:         0 (0.0%)
   Результаты:     df_res_with_qwen_answers.csv

🔍 Примеры ответов:

Строка 0:
   Ответ: Упрощение исследований поэтических текстов...

Строка 1:
   Ответ: Объединение существующих реализаций алгоритмов для численного моделирования цунами в одну систему, к...

Строка 2:
   Ответ: Генотипирования вируса клещевого энцефалита (ВКЭ)....

Строка 3:
   Ответ: Решена задача геонавигации нефтегазовых скважин с горизонтальным завершением....

Строка 4:
   Ответ: Предложенный подход к формализации структуры ЕИП предприятия-разработчика МЭС....


In [21]:
df_res_qwen = pd.read_csv('/home/pmartynyuk/UnTIE project/scripts/rus_df_res_with_qwen_answers.csv',
                            encoding='utf-8', sep=',')
df_res_qwen.head()

,text,Task_aspects,Task_count,text_clean,qwen2.5-3B_answer,time
0,В рамках проекта по <Task> автоматизации раб...,['автоматизации работы с поэтическими текстами...,2,В рамках проекта по автоматизации работы с ...,Упрощение исследований поэтических текстов,1.056020
1,Проблема <Task> своевременного предупреждения...,['своевременного предупреждения населения об о...,2,Проблема своевременного предупреждения насел...,Объединение существующих реализаций алгоритмов...,3.587075
2,Работа посвящена <Contrib> описанию структур...,['генотипирования вируса клещевого энцефалита ...,2,Работа посвящена описанию структуры информа...,Генотипирования вируса клещевого энцефалита (В...,1.354576
3,Для эффективного <Task> освоения залежей угл...,['освоения залежей углеводородов'],1,Для эффективного освоения залежей углеводор...,Решена задача геонавигации нефтегазовых скважи...,1.397231
4,При <Task> формировании единого информационно...,['формировании единого информационного простра...,1,При формировании единого информационного про...,Предложенный подход к формализации структуры Е...,1.547555


In [22]:
from transformers import *
roberta_qa_dir = "/home/pmartynyuk/UnTIE project/scripts/models_processing/models/rubert_ru_qa_model"

model = AutoModelForQuestionAnswering.from_pretrained(roberta_qa_dir, output_attentions=True).to(device)
concrete_tokenizer = AutoTokenizer.from_pretrained(roberta_qa_dir)

loading configuration file /home/pmartynyuk/UnTIE project/scripts/models_processing/models/rubert_ru_qa_model/config.json
Model config XLMRobertaConfig {
  "architectures": [
    "XLMRobertaForQuestionAnswering"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "language": "english",
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "xlm-roberta",
  "name": "XLMRoberta",
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "output_attentions": true,
  "output_past": true,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.53.3",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 250002
}

loading weights file /home/pmartynyuk/UnTIE project/scri

In [23]:
from transformers import pipeline
import torch

# Загрузка модели (весит всего ~50 МБ)
qa_pipeline = pipeline(
    "question-answering",
    model=model,
    tokenizer=concrete_tokenizer,
    device=1 if torch.cuda.is_available() else -1,  # -1 = CPU
    max_seq_len=512,      # максимальная длина чанка
    doc_stride=128,       # перекрытие между чанками (рекомендуется 128)
    max_answer_len=100    # макс. длина извлекаемого ответа
)

Device set to use cuda:1


In [28]:
idx = 0

context = str(df_res_qwen.loc[idx, 'text_clean'])
question = "Какая задача была поставлена?"

In [29]:
result = qa_pipeline(
            question=question,
            context=context,
            handle_impossible_answer=True,
            max_seq_len=512,
            doc_stride=128
        )
result

{'score': 0.2592003643512726,
 'start': 895,
 'end': 937,
 'answer': ' призвана решать информационная система  ,'}

In [34]:
def answer_question_rubert(context: str, question: str) -> str:
    """
    Надёжная функция для rubert QA. Всегда возвращает непустую строку.
    """
    # === 1. Валидация входных данных ===
    if not isinstance(context, str) or not context.strip():
        return "ERROR: пустой или некорректный текст"
    if not isinstance(question, str) or not question.strip():
        return "ERROR: пустой вопрос"
    
    # === 2. Вызов модели ===
    try:
        result = qa_pipeline(
            question=question,
            context=context,
            handle_impossible_answer=True,
            max_seq_len=512,
            doc_stride=128
        )
    except Exception as e:
        return f"ERROR: pipeline exception ({type(e).__name__})"
    
    # === 3. Обработка результата ===
    try:
        # Список ответов (длинный текст)
        if isinstance(result, list) and result:
            # Фильтруем только словари с ответами
            valid_answers = [
                r for r in result 
                if isinstance(r, dict) 
                and 'answer' in r 
                and isinstance(r['answer'], str)
                # and len(r['answer'].strip()) >= 2  # минимум 2 символа
                # and r.get('score', 0) > 0.001       # любой ненулевой скор
            ]
            
            if valid_answers:
                # Берём ответ с максимальным скором
                best = max(valid_answers, key=lambda x: x.get('score', 0))
                answer = best['answer'].strip()
                # Ограничиваем длину ответа
                return answer[:150] if answer else "Ответ не найден"
            else:
                return "Ответ не найден"
        
        # Один ответ (короткий текст)
        elif isinstance(result, dict) and 'answer' in result:
            answer = str(result.get('answer', '')).strip()
            if len(answer) >= 2 and result.get('score', 0) > 0.001:
                return answer[:150]
            else:
                return "Ответ не найден"
        
        # Любой другой случай
        else:
            return "Ответ не найден"
    
    except Exception as e:
        return f"ERROR: processing exception ({type(e).__name__})"

In [33]:
df_res_rubert = df_res.drop(columns=["qwen2.5-3B_answer", "time"])

In [38]:
import pandas as pd
import time
from tqdm import tqdm

# === КРИТИЧЕСКИ ВАЖНО: Очистка колонки от существующих NaN ===
if 'rubert_answer' in df_res_rubert.columns:
    df_res_rubert['rubert_answer'] = df_res_rubert['rubert_answer'].fillna("").astype(str)
else:
    df_res_rubert['rubert_answer'] = ""

# Убедимся, что original_text не содержит NaN
df_res_rubert['text_clean'] = df_res_rubert['text_clean'].fillna("")

question = "Which task was solved?"

pbar = tqdm(total=len(df_res_rubert), desc="rubert QA", unit="текст", ncols=80)

for idx in range(len(df_res_rubert)):
    # Пропускаем уже обработанные (не "Ответ не найден" и не ошибки)
    current = str(df_res_rubert.loc[idx, 'rubert_answer']).strip()
    if current and not current.startswith(("Ответ не найден", "ERROR")):
        pbar.update(1)
        continue
    
    context = str(df_res_rubert.loc[idx, 'text_clean'])
    
    # Получаем ответ (гарантированно строка)
    answer = answer_question_rubert(context, question)
    
    # Принудительно сохраняем как строку
    df_res_rubert.loc[idx, 'rubert_answer'] = str(answer)
    
    pbar.update(1)
    
    # Промежуточное сохранение
    if (idx + 1) % 10 == 0:
        df_res_rubert.to_csv('df_res_rubert.csv', index=False)
        pbar.set_postfix({"последний": answer[:30]})

pbar.close()
df_res_rubert.to_csv('df_res_rubert.csv', index=False)

# Финальная статистика
total = len(df_res_rubert)
not_found = df_res_rubert['rubert_answer'].str.contains("Ответ не найден", na=False).sum()
errors = df_res_rubert['rubert_answer'].str.startswith("ERROR", na=False).sum()
valid = total - not_found - errors
nan_check = df_res_rubert['rubert_answer'].isna().sum()

print("\n" + "="*70)
print("✅ ОБРАБОТКА ЗАВЕРШЕНА")
print(f"   Всего строк:          {total}")
print(f"   Валидных ответов:     {valid} ({valid/total*100:.1f}%)")
print(f"   'Ответ не найден':    {not_found} ({not_found/total*100:.1f}%)")
print(f"   Ошибок:               {errors} ({errors/total*100:.1f}%)")
print(f"   NaN в колонке:        {nan_check} ← ДОЛЖНО БЫТЬ 0")
print("="*70)

# Примеры хороших ответов
print("\n🔍 Примеры найденных ответов:")
valid_answers = df_res_rubert[~df_res_rubert['rubert_answer'].str.contains("Ответ не найден|ERROR", na=False)]
for i, (_, row) in enumerate(valid_answers.head(5).iterrows()):
    print(f"{i+1}. {row['rubert_answer']}")

rubert QA:   2%|▋                            | 4/168 [00:00<00:04, 34.76текст/s]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
rubert QA: 100%|█| 168/168 [00:05<00:00, 31.95текст/s, последний=Показаны недост


✅ ОБРАБОТКА ЗАВЕРШЕНА
   Всего строк:          168
   Валидных ответов:     168 (100.0%)
   'Ответ не найден':    0 (0.0%)
   Ошибок:               0 (0.0%)
   NaN в колонке:        0 ← ДОЛЖНО БЫТЬ 0

🔍 Примеры найденных ответов:
1. призвана решать информационная система  ,
2. своевременного предупреждения населения об опасности цунами
3. генотипирования вируса клещевого энцефалита (ВКЭ)  .
4. геонавигация скважины со сложной траекторией выполняется по геофизическим данным в реальном масштабе времени. Представленная работа посвящена   разраб
5. с использованием метода матрицы структуры проекта,


In [40]:
df_res_rubert = pd.read_csv('/home/pmartynyuk/UnTIE project/scripts/df_res_rubert.csv',
                            encoding='utf-8', sep=',')
df_res_rubert.head()

,text,Task_aspects,Task_count,text_clean,rubert_answer
0,В рамках проекта по <Task> автоматизации раб...,['автоматизации работы с поэтическими текстами...,2,В рамках проекта по автоматизации работы с ...,"призвана решать информационная система ,"
1,Проблема <Task> своевременного предупреждения...,['своевременного предупреждения населения об о...,2,Проблема своевременного предупреждения насел...,своевременного предупреждения населения об опа...
2,Работа посвящена <Contrib> описанию структур...,['генотипирования вируса клещевого энцефалита ...,2,Работа посвящена описанию структуры информа...,генотипирования вируса клещевого энцефалита (В...
3,Для эффективного <Task> освоения залежей угл...,['освоения залежей углеводородов'],1,Для эффективного освоения залежей углеводор...,геонавигация скважины со сложной траекторией в...
4,При <Task> формировании единого информационно...,['формировании единого информационного простра...,1,При формировании единого информационного про...,с использованием метода матрицы структуры прое...
